# Solar Filament Instance Segmentation — Reproducible Pipeline

This notebook demonstrates the end-to-end pipeline using the reusable `solar_filament` package rather than duplicating logic in notebook cells.

For the full exploratory investigation, failed experiments, and original Colab A100 outputs, see [`01_original_investigation.ipynb`](01_original_investigation.ipynb).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from solar_filament.config import load_config
from solar_filament.utils import resolve_device, seed_everything

cfg = load_config(REPO_ROOT / 'configs' / 'default.yaml')
seed_everything(cfg.seed)
device = resolve_device(cfg.device)
print('Device:', device)
print('Patch config:', cfg.patches)

## 1. Dataset & Patch Coverage Audit

Point `cfg.paths` to your local `MAGFiLO_1.0_Kaggle_2026` directory before running the cells below.

In [ ]:
import json
from solar_filament.data import (
    AnnotationIndex,
    audit_annotations,
    audit_patch_coverage,
    group_records_by_filename,
    split_by_filename,
)

index = AnnotationIndex.from_json(cfg.paths.train_annotations_json)
ann_report = audit_annotations(index, check_mask_overlaps=True)
patch_report = audit_patch_coverage(
    index,
    image_size=cfg.data.image_size,
    patch_size=cfg.patches.patch_size,
    overlap=cfg.patches.overlap,
)
print(json.dumps(ann_report, indent=2))
print(json.dumps(patch_report, indent=2))

## 2. Leakage-Free Split & Dataset Construction

In [ ]:
from torch.utils.data import DataLoader
from solar_filament.data import SolarFilamentDataset, detection_collate_fn

filename_to_ids = group_records_by_filename(index)
train_files, val_files = split_by_filename(
    filename_to_ids,
    val_size=cfg.data.val_size,
    random_state=cfg.data.split_seed,
)
print(f'Train images: {len(train_files)} | Val images: {len(val_files)}')

train_ds = SolarFilamentDataset(
    filenames=train_files,
    filename_to_ids=filename_to_ids,
    annotation_index=index,
    image_dir=cfg.paths.train_images_dir,
    image_size=cfg.data.image_size,
    patch_size=cfg.patches.patch_size,
    overlap=cfg.patches.overlap,
    training=True,
    cache_size=cfg.data.cache_size,
    seed=cfg.seed,
)
val_ds = SolarFilamentDataset(
    filenames=val_files,
    filename_to_ids=filename_to_ids,
    annotation_index=index,
    image_dir=cfg.paths.train_images_dir,
    image_size=cfg.data.image_size,
    patch_size=cfg.patches.patch_size,
    overlap=cfg.patches.overlap,
    training=False,
    cache_size=cfg.data.cache_size,
    seed=cfg.seed,
)
print(f'Train patches: {len(train_ds):,} | Val patches: {len(val_ds):,}')

## 3. Model Initialization & Training (FP32)

In [ ]:
from solar_filament.models import build_mask_rcnn
from solar_filament.training import train_model

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.training.batch_size,
    shuffle=True,
    num_workers=cfg.data.num_workers,
    pin_memory=(cfg.data.pin_memory and device.type == 'cuda'),
    collate_fn=detection_collate_fn,
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.training.batch_size,
    shuffle=False,
    num_workers=cfg.data.num_workers,
    pin_memory=(cfg.data.pin_memory and device.type == 'cuda'),
    collate_fn=detection_collate_fn,
)

model = build_mask_rcnn(cfg.model, pretrained=cfg.model.pretrained_coco)
# Uncomment to run training:
# history = train_model(model, train_loader, val_loader, cfg, device)

## 4. Tiled Inference & Submission Validation

In [ ]:
from solar_filament.models import load_model_checkpoint
from solar_filament.inference import run_directory_inference
from solar_filament.submission import (
    build_submission_rows,
    validate_submission_csv,
    write_submission_csv,
)

# Uncomment once a trained checkpoint exists at cfg.paths.best_checkpoint:
# model = build_mask_rcnn(cfg.model, pretrained=False)
# load_model_checkpoint(model, cfg.paths.best_checkpoint, device=device)
# results = run_directory_inference(model, cfg.paths.test_images_dir, cfg, device)
# rows = build_submission_rows(results)
# write_submission_csv(rows, cfg.paths.submission_csv)
# summary = validate_submission_csv(
#     cfg.paths.submission_csv,
#     image_size=cfg.data.image_size,
#     min_mask_area=cfg.inference.min_mask_area,
#     test_dir=cfg.paths.test_images_dir,
# )
# print(summary)